# exp053 Inference NB (Kaggle CPU): Stage 2A → submission.csv

**Input**:
- birdclef-2026 (competition)
- maekeso/birdclef2026-exp053-stage2a-effv2s (Stage 2A ckpt)

**Output**: submission.csv

**Approach**:
- pytorch CPU inference (no ONNX、Stage 2A 22M params で CPU 内収まる)
- batch 32 chunks 同時推論
- TopN N=1 PP (paper 256 spec、exp048 0.950 で実証)

**Expected runtime**: 40-60 min (CPU 90 min 制限内)
**Expected LB**: 0.88-0.92 standalone


In [ ]:
# ============================================================
# Cell 1: Setup
# ============================================================
import os, sys, json, time, glob
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import soundfile as sf
import librosa
import timm
from tqdm.auto import tqdm

print(f"torch: {torch.__version__}")
print(f"cuda: {torch.cuda.is_available()} (expected False for submission NB)")
DEVICE = "cpu"

torch.set_num_threads(4)
torch.set_grad_enabled(False)
print(f"torch.num_threads: {torch.get_num_threads()}")

# Paths
COMP_ROOT = Path("/kaggle/input/competitions/birdclef-2026")
if not COMP_ROOT.exists():
    COMP_ROOT = Path("/kaggle/input/birdclef-2026")
assert COMP_ROOT.exists(), f"BC2026 not mounted at {COMP_ROOT}"

# Stage 2A ckpt - try multiple paths
CKPT_CANDIDATES = [
    Path("/kaggle/input/birdclef2026-exp053-stage2a-effv2s/stage2a_best.pth"),
    Path("/kaggle/input/datasets/maekeso/birdclef2026-exp053-stage2a-effv2s/stage2a_best.pth"),
]
CKPT_PATH = next((p for p in CKPT_CANDIDATES if p.exists()), None)
assert CKPT_PATH is not None, f"Stage 2A ckpt not found in: {CKPT_CANDIDATES}"
print(f"Stage 2A ckpt: {CKPT_PATH} ({CKPT_PATH.stat().st_size/1e6:.1f} MB)")

TEST_DIR = COMP_ROOT / "test_soundscapes"
print(f"test_soundscapes: {TEST_DIR}")


In [ ]:
# ============================================================
# Cell 2: 234 species list (embedded)
# ============================================================
__BC2026_SPECIES = [
    ('Guyalna cuta', '1161364', 'Insecta'),
    ('Caiman yacare', '116570', 'Reptilia'),
    ('Leptodactylus luctator', '1176823', 'Amphibia'),
    ('Adenomera guarani', '1491113', 'Amphibia'),
    ('Lysapsus limellum', '1595929', 'Amphibia'),
    ('Equus caballus', '209233', 'Mammalia'),
    ('Leptodactylus syphax', '22930', 'Amphibia'),
    ('Leptodactylus mystacinus', '22956', 'Amphibia'),
    ('Leptodactylus podicipinus', '22961', 'Amphibia'),
    ('Leptodactylus elenae', '22967', 'Amphibia'),
    ('Leptodactylus fuscus', '22973', 'Amphibia'),
    ('Leptodactylus labyrinthicus', '22983', 'Amphibia'),
    ('Leptodactylus petersii', '22985', 'Amphibia'),
    ('Physalaemus centralis', '23150', 'Amphibia'),
    ('Physalaemus albifrons', '23154', 'Amphibia'),
    ('Physalaemus albonotatus', '23158', 'Amphibia'),
    ('Pseudopaludicola mystacalis', '23176', 'Amphibia'),
    ('Phyllomedusa sauvagii', '23724', 'Amphibia'),
    ('Scinax nasicus', '24279', 'Amphibia'),
    ('Scinax fuscovarius', '24285', 'Amphibia'),
    ('Scinax fuscomarginatus', '24287', 'Amphibia'),
    ('Scinax acuminatus', '24321', 'Amphibia'),
    ('Quesada gigas', '244024', 'Insecta'),
    ('Chiasmocleis mehelyi', '25073', 'Amphibia'),
    ('Elachistocleis bicolor', '25092', 'Amphibia'),
    ('Dermatonotus muelleri', '25214', 'Amphibia'),
    ('Physalaemus biligonigerus', '326272', 'Amphibia'),
    ('Panthera onca', '41970', 'Mammalia'),
    ('Alouatta caraya', '43435', 'Mammalia'),
    ('Canis familiaris', '47144', 'Mammalia'),
    ('Insect son01', '47158son01', 'Insecta'),
    ('Insect son02', '47158son02', 'Insecta'),
    ('Insect son03', '47158son03', 'Insecta'),
    ('Insect son04', '47158son04', 'Insecta'),
    ('Insect son05', '47158son05', 'Insecta'),
    ('Insect son06', '47158son06', 'Insecta'),
    ('Insect son07', '47158son07', 'Insecta'),
    ('Insect son08', '47158son08', 'Insecta'),
    ('Insect son09', '47158son09', 'Insecta'),
    ('Insect son10', '47158son10', 'Insecta'),
    ('Insect son11', '47158son11', 'Insecta'),
    ('Insect son12', '47158son12', 'Insecta'),
    ('Insect son13', '47158son13', 'Insecta'),
    ('Insect son14', '47158son14', 'Insecta'),
    ('Insect son15', '47158son15', 'Insecta'),
    ('Insect son16', '47158son16', 'Insecta'),
    ('Insect son17', '47158son17', 'Insecta'),
    ('Insect son18', '47158son18', 'Insecta'),
    ('Insect son19', '47158son19', 'Insecta'),
    ('Insect son20', '47158son20', 'Insecta'),
    ('Insect son21', '47158son21', 'Insecta'),
    ('Insect son22', '47158son22', 'Insecta'),
    ('Insect son23', '47158son23', 'Insecta'),
    ('Insect son24', '47158son24', 'Insecta'),
    ('Insect son25', '47158son25', 'Insecta'),
    ('Physalaemus nattereri', '476521', 'Amphibia'),
    ('Sapajus cay', '516975', 'Mammalia'),
    ('Pithecopus azureus', '517063', 'Amphibia'),
    ('Boana lundii', '555123', 'Amphibia'),
    ('Boana punctata', '555145', 'Amphibia'),
    ('Boana raniceps', '555146', 'Amphibia'),
    ('Ameerega picta', '64898', 'Amphibia'),
    ('Dendropsophus minutus', '65377', 'Amphibia'),
    ('Dendropsophus nanus', '65380', 'Amphibia'),
    ('Pseudis platensis', '66971', 'Amphibia'),
    ('Rhinella diptycha', '67107', 'Amphibia'),
    ('Trachycephalus typhonius', '67252', 'Amphibia'),
    ('Leptodactylus macrosternum', '70711', 'Amphibia'),
    ('Plecturocebus pallescens', '738183', 'Mammalia'),
    ('Bos taurus', '74113', 'Mammalia'),
    ('Mico melanurus', '74580', 'Mammalia'),
    ('Prionacris erosa', '760266', 'Insecta'),
    ('Hylophilus pectoralis', 'ashgre1', 'Aves'),
    ('Mustelirallus albicollis', 'astcra1', 'Aves'),
    ('Crax fasciolata', 'bafcur1', 'Aves'),
    ('Micrastur ruficollis', 'baffal1', 'Aves'),
    ('Coereba flaveola', 'banana', 'Aves'),
    ('Thamnophilus doliatus', 'barant1', 'Aves'),
    ('Procnias nudicollis', 'batbel1', 'Aves'),
    ('Ara ararauna', 'baymac', 'Aves'),
    ('Dendrocygna autumnalis', 'bbwduc', 'Aves'),
    ('Microspingus melanoleucus', 'bcwfin2', 'Aves'),
    ('Donacobius atricapilla', 'bkcdon', 'Aves'),
    ('Aratinga nenday', 'bkhpar', 'Aves'),
    ('Busarellus nigricollis', 'blchaw1', 'Aves'),
    ('Spizaetus tyrannus', 'blheag1', 'Aves'),
    ('Tityra cayana', 'blttit1', 'Aves'),
    ('Myiarchus tyrannulus', 'bncfly', 'Aves'),
    ('Megarynchus pitangua', 'bobfly1', 'Aves'),
    ('Progne tapera', 'brcmar1', 'Aves'),
    ('Tyto furcata', 'brnowl', 'Aves'),
    ('Momotus momota', 'bucmot4', 'Aves'),
    ('Thectocercus acuticaudatus', 'bucpar', 'Aves'),
    ('Amazona aestiva', 'bufpar', 'Aves'),
    ('Theristicus caudatus', 'bunibi1', 'Aves'),
    ('Athene cunicularia', 'burowl', 'Aves'),
    ('Colaptes campestris', 'camfli1', 'Aves'),
    ('Ortalis canicollis', 'chacha1', 'Aves'),
    ('Mimus saturninus', 'chbmoc1', 'Aves'),
    ('Gnorimopsar chopi', 'chobla1', 'Aves'),
    ('Conirostrum speciosum', 'chvcon1', 'Aves'),
    ('Synallaxis hypospodia', 'cibspi1', 'Aves'),
    ('Micrastur semitorquatus', 'coffal1', 'Aves'),
    ('Nyctidromus albicollis', 'compau', 'Aves'),
    ('Nyctibius griseus', 'compot1', 'Aves'),
    ('Turdus amaurochalinus', 'crbthr1', 'Aves'),
    ('Pachyramphus validus', 'crebec1', 'Aves'),
    ('Taoniscus nanus', 'dwatin1', 'Aves'),
    ('Icterus pyrrhopterus', 'epaori4', 'Aves'),
    ('Lathrotriccus euleri', 'eulfly1', 'Aves'),
    ('Cantorchilus guarayanus', 'fabwre1', 'Aves'),
    ('Glaucidium brasilianum', 'fepowl', 'Aves'),
    ('Machaeropterus pyrocephalus', 'ficman1', 'Aves'),
    ('Myiothlypis flaveola', 'flawar1', 'Aves'),
    ('Tyrannus savana', 'fotfly', 'Aves'),
    ('Cnemotriccus fuscatus', 'fusfly1', 'Aves'),
    ('Hylocharis chrysura', 'gilhum1', 'Aves'),
    ('Aramides ypecaha', 'giwrai1', 'Aves'),
    ('Chionomesa fimbriata', 'glteme1', 'Aves'),
    ('Saltator coerulescens', 'grasal3', 'Aves'),
    ('Crotophaga major', 'greani1', 'Aves'),
    ('Taraba major', 'greant1', 'Aves'),
    ('Myiopagis viridicata', 'greela', 'Aves'),
    ('Pitangus sulphuratus', 'grekis', 'Aves'),
    ('Nyctibius grandis', 'grepot1', 'Aves'),
    ('Phacellodomus ruber', 'gretho2', 'Aves'),
    ('Tringa melanoleuca', 'greyel', 'Aves'),
    ('Leptotila rufaxilla', 'grfdov1', 'Aves'),
    ('Eucometis penicillata', 'grhtan1', 'Aves'),
    ('Aramides cajaneus', 'gycwor1', 'Aves'),
    ('Anhima cornuta', 'horscr1', 'Aves'),
    ('Passer domesticus', 'houspa', 'Aves'),
    ('Anodorhynchus hyacinthinus', 'hyamac1', 'Aves'),
    ('Elaenia spectabilis', 'larela1', 'Aves'),
    ('Elaenia chiriquensis', 'lesela1', 'Aves'),
    ('Emberizoides ypiranganus', 'lesgrf1', 'Aves'),
    ('Aramus guarauna', 'limpki', 'Aves'),
    ('Dryocopus lineatus', 'linwoo1', 'Aves'),
    ('Coccycua minuta', 'litcuc2', 'Aves'),
    ('Setopagis parvula', 'litnig1', 'Aves'),
    ('Pyrrhura frontalis', 'mabpar', 'Aves'),
    ('Cercomacra melanaria', 'magant1', 'Aves'),
    ('Cissopis leverianus', 'magtan2', 'Aves'),
    ('Polioptila dumicola', 'masgna1', 'Aves'),
    ('Chordeiles nacunda', 'nacnig1', 'Aves'),
    ('Rufirallus schomburgkii', 'ocecra1', 'Aves'),
    ('Sittasomus griseicapillus', 'oliwoo1', 'Aves'),
    ('Icterus croconotus', 'orbtro3', 'Aves'),
    ('Amazona amazonica', 'orwpar', 'Aves'),
    ('Pandion haliaetus', 'osprey', 'Aves'),
    ('Synallaxis albescens', 'pabspi1', 'Aves'),
    ('Furnarius leucopus', 'palhor3', 'Aves'),
    ('Thraupis palmarum', 'paltan1', 'Aves'),
    ('Dromococcyx phasianellus', 'phecuc1', 'Aves'),
    ('Patagioenas picazuro', 'picpig2', 'Aves'),
    ('Legatus leucophaius', 'pirfly1', 'Aves'),
    ('Thamnophilus pelzelni', 'plasla1', 'Aves'),
    ('Inezia inornata', 'platyr1', 'Aves'),
    ('Cyanocorax chrysops', 'plcjay1', 'Aves'),
    ('Theristicus caerulescens', 'pluibi1', 'Aves'),
    ('Cyanocorax cyanomelas', 'purjay1', 'Aves'),
    ('Hemitriccus margaritaceiventer', 'pvttyr1', 'Aves'),
    ('Ara chloropterus', 'ragmac1', 'Aves'),
    ('Campylorhamphus trochilirostris', 'rebscy1', 'Aves'),
    ('Coryphospingus cucullatus', 'recfin1', 'Aves'),
    ('Gallus gallus', 'redjun', 'Aves'),
    ('Cariama cristata', 'relser1', 'Aves'),
    ('Megaceryle torquata', 'rinkin1', 'Aves'),
    ('Myiothlypis rivularis', 'rivwar1', 'Aves'),
    ('Rupornis magnirostris', 'roahaw', 'Aves'),
    ('Turdus rufiventris', 'rubthr1', 'Aves'),
    ('Pseudoseisura unirufa', 'rufcac2', 'Aves'),
    ('Casiornis rufus', 'rufcas2', 'Aves'),
    ('Conopophaga lineata', 'rufgna3', 'Aves'),
    ('Furnarius rufus', 'rufhor2', 'Aves'),
    ('Antrostomus rufus', 'rufnig1', 'Aves'),
    ('Phacellodomus rufifrons', 'ruftho1', 'Aves'),
    ('Poecilotriccus latirostris', 'ruftof1', 'Aves'),
    ('Myiozetetes cayanensis', 'rumfly1', 'Aves'),
    ('Tigrisoma lineatum', 'ruther1', 'Aves'),
    ('Galbula ruficauda', 'rutjac1', 'Aves'),
    ('Arremon flavirostris', 'sabspa1', 'Aves'),
    ('Sicalis flaveola', 'saffin', 'Aves'),
    ('Thraupis sayaca', 'saytan1', 'Aves'),
    ('Columbina squammata', 'scadov1', 'Aves'),
    ('Pionus maximiliani', 'schpar1', 'Aves'),
    ('Phaethornis eurynome', 'scther1', 'Aves'),
    ('Myiarchus ferox', 'shcfly1', 'Aves'),
    ('Accipiter striatus', 'shshaw', 'Aves'),
    ('Lurocalis semitorquatus', 'shtnig1', 'Aves'),
    ('Ramphocelus carbo', 'sibtan2', 'Aves'),
    ('Crotophaga ani', 'smbani', 'Aves'),
    ('Crypturellus parvirostris', 'smbtin1', 'Aves'),
    ('Cacicus solitarius', 'sobcac1', 'Aves'),
    ('Camptostoma obsoletum', 'sobtyr1', 'Aves'),
    ('Myiozetetes similis', 'socfly1', 'Aves'),
    ('Synallaxis frontalis', 'sofspi1', 'Aves'),
    ('Corythopis delalandi', 'souant1', 'Aves'),
    ('Vanellus chilensis', 'soulap1', 'Aves'),
    ('Chauna torquata', 'souscr1', 'Aves'),
    ('Hypoedaleus guttatus', 'spbant3', 'Aves'),
    ('Synallaxis spixi', 'spispi1', 'Aves'),
    ('Antiurus maculicaudus', 'sptnig1', 'Aves'),
    ('Piaya cayana', 'squcuc1', 'Aves'),
    ('Dendroplex picus', 'stbwoo2', 'Aves'),
    ('Tapera naevia', 'strcuc1', 'Aves'),
    ('Butorides striata', 'strher2', 'Aves'),
    ('Asio clamator', 'strowl1', 'Aves'),
    ('Eupetomena macroura', 'swthum1', 'Aves'),
    ('Chiroxiphia caudata', 'swtman1', 'Aves'),
    ('Crypturellus tataupa', 'tattin1', 'Aves'),
    ('Campylorhynchus turdinus', 'thlwre1', 'Aves'),
    ('Ramphastos toco', 'toctou1', 'Aves'),
    ('Tyrannus melancholicus', 'trokin', 'Aves'),
    ('Megascops choliba', 'trsowl', 'Aves'),
    ('Crypturellus undulatus', 'undtin1', 'Aves'),
    ('Thamnophilus caerulescens', 'varant1', 'Aves'),
    ('Jacana jacana', 'watjac1', 'Aves'),
    ('Pyriglena maura', 'wesfie1', 'Aves'),
    ('Dendrocygna viduata', 'wfwduc1', 'Aves'),
    ('Biatas nigropectus', 'whbant2', 'Aves'),
    ('Myiothlypis leucoblephara', 'whbwar2', 'Aves'),
    ('Melanerpes candidus', 'whiwoo1', 'Aves'),
    ('Synallaxis albilora', 'whlspi1', 'Aves'),
    ('Cyanocorax cyanopogon', 'whnjay1', 'Aves'),
    ('Leptotila verreauxi', 'whtdov', 'Aves'),
    ('Picumnus albosquamatus', 'whwpic1', 'Aves'),
    ('Caracara plancus', 'y00678', 'Aves'),
    ('Paroaria capitata', 'yebcar', 'Aves'),
    ('Elaenia flavogaster', 'yebela1', 'Aves'),
    ('Primolius auricollis', 'yecmac', 'Aves'),
    ('Brotogeris chiriri', 'yecpar', 'Aves'),
    ('Daptrius chimachima', 'yehcar1', 'Aves'),
    ('Tolmomyias sulphurescens', 'yeofly1', 'Aves'),
]
PRIMARY_LABELS = [r[1] for r in __BC2026_SPECIES]
N_CLASSES = len(PRIMARY_LABELS)
print(f"BC2026 species: {N_CLASSES}")


In [ ]:
# ============================================================
# Cell 3: Config
# ============================================================
class CFG:
    BACKBONE = "tf_efficientnetv2_s.in21k_ft_in1k"
    N_CLASSES = N_CLASSES

    SR = 32000
    CHUNK_SEC = 5
    CHUNK_LEN = SR * CHUNK_SEC  # 160000

    N_MELS = 128
    N_FFT = 2048
    HOP = 512
    FMIN = 20
    FMAX = 16000

    BATCH_SIZE = 32  # CPU batch

    # PP
    FCS_TOP_K = 1     # paper 256 spec
    FCS_POWER = 1.0   # exp048 0.950 で実証

print(f"Mel: {CFG.N_MELS}×T, {CFG.SR}Hz")
print(f"PP: TopN N=1, POWER=1.0 (paper 256)")


In [ ]:
# ============================================================
# Cell 4: Model definition + load Stage 2A ckpt
# ============================================================
import torchaudio

class MelExtractor(nn.Module):
    def __init__(self):
        super().__init__()
        self.mel = torchaudio.transforms.MelSpectrogram(
            sample_rate=CFG.SR, n_fft=CFG.N_FFT, hop_length=CFG.HOP,
            n_mels=CFG.N_MELS, f_min=CFG.FMIN, f_max=CFG.FMAX,
        )
        self.db = torchaudio.transforms.AmplitudeToDB(top_db=80.0)

    def forward(self, wav):
        mel = self.mel(wav)
        mel = self.db(mel)
        mel = torch.clamp(mel, -80.0, 0.0)
        mel = (mel + 40.0) / 40.0
        return mel


class SEDHead(nn.Module):
    def __init__(self, in_dim, n_classes):
        super().__init__()
        self.att = nn.Linear(in_dim, n_classes)
        self.cla = nn.Linear(in_dim, n_classes)

    def forward(self, x):
        att = torch.tanh(self.att(x))
        cla = self.cla(x)
        norm_att = F.softmax(att, dim=1)
        clipwise = (norm_att * cla).sum(dim=1)
        return clipwise


class SEDModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = timm.create_model(
            CFG.BACKBONE, pretrained=False, in_chans=3,
            num_classes=0, global_pool="",
        )
        feat_dim = self.backbone.num_features
        self.head = SEDHead(feat_dim, CFG.N_CLASSES)

    def forward(self, mel):
        x = mel.unsqueeze(1).repeat(1, 3, 1, 1)
        feat = self.backbone(x)
        feat = feat.mean(dim=2)
        feat = feat.transpose(1, 2)
        return self.head(feat)


# Load Stage 2A ckpt
model = SEDModel()
ckpt = torch.load(CKPT_PATH, map_location="cpu", weights_only=False)
state = ckpt.get("state_dict", ckpt)
# Remove "module." prefix if DataParallel was used
state = {k.replace("module.", ""): v for k, v in state.items()}
missing, unexpected = model.load_state_dict(state, strict=False)
print(f"Missing keys: {len(missing)}")
print(f"Unexpected keys: {len(unexpected)}")
print(f"Stage 2A val_ns22: {ckpt.get('val_ns22', 'n/a')}")
print(f"Stage 2A val_macro: {ckpt.get('val_macro', 'n/a')}")
model.eval()
model = model.to(DEVICE)
mel_extractor = MelExtractor().to(DEVICE)
print(f"Model loaded, params: {sum(p.numel() for p in model.parameters())/1e6:.1f}M")


In [ ]:
# ============================================================
# Cell 5: Discover test_soundscapes
# ============================================================
test_files = sorted(list(TEST_DIR.glob("*.ogg")))
print(f"Test soundscapes: {len(test_files)}")
if test_files:
    print(f"  Sample: {test_files[0].name}")
    # Check duration of first file
    info = sf.info(str(test_files[0]))
    print(f"  Duration: {info.duration:.1f}s, SR: {info.samplerate}, Channels: {info.channels}")


In [ ]:
# ============================================================
# Cell 6: Inference function (per file, returns per-segment predictions)
# ============================================================
@torch.no_grad()
def predict_file(file_path, model, mel_ex):
    """Process one test_soundscape file → list of (row_id, 234 probs)"""
    file_id = Path(file_path).stem

    # Load audio
    try:
        wav, sr = sf.read(str(file_path), dtype="float32")
        if wav.ndim > 1:
            wav = wav.mean(axis=1)
        if sr != CFG.SR:
            wav = librosa.resample(wav, orig_sr=sr, target_sr=CFG.SR)
    except Exception as e:
        print(f"  [read err] {file_path}: {str(e)[:80]}")
        return []

    # Segment into 5s chunks (BC2026 standard: 60s file → 12 segments)
    n_full_chunks = len(wav) // CFG.CHUNK_LEN
    if n_full_chunks == 0:
        # Pad to single chunk
        wav = np.pad(wav, (0, CFG.CHUNK_LEN - len(wav)))
        n_full_chunks = 1
        chunks = [wav]
    else:
        chunks = [wav[i*CFG.CHUNK_LEN:(i+1)*CFG.CHUNK_LEN] for i in range(n_full_chunks)]

    # Build batch tensor
    batch_wav = torch.from_numpy(np.stack(chunks)).float()

    # Batched inference (handle large batch via chunking)
    all_preds = []
    for i in range(0, len(batch_wav), CFG.BATCH_SIZE):
        sub_batch = batch_wav[i:i+CFG.BATCH_SIZE]
        mel = mel_ex(sub_batch)
        logit = model(mel)
        prob = torch.sigmoid(logit).numpy()
        all_preds.append(prob)
    probs = np.concatenate(all_preds, axis=0)  # (n_chunks, 234)

    # Build row_ids: {file_id}_{end_time_seconds}
    rows = []
    for i, p in enumerate(probs):
        end_sec = (i + 1) * CFG.CHUNK_SEC
        row_id = f"{file_id}_{end_sec}"
        row = {"row_id": row_id}
        for j, lbl in enumerate(PRIMARY_LABELS):
            row[lbl] = float(p[j])
        rows.append(row)
    return rows


In [ ]:
# ============================================================
# Cell 7: Run inference on all test files (handle empty test_soundscapes)
# ============================================================
print(f"Starting inference on {len(test_files)} files...")

if len(test_files) == 0:
    print("\n⚠️ test_soundscapes is empty (expected for Save Version)")
    print("   実 test data は Kaggle Submit 時に自動配置される")
    print("   この cell では empty sub_df 作成、submission.csv は dummy 出力")
    all_rows = []
else:
    start_t = time.time()
    all_rows = []
    for i, f in enumerate(tqdm(test_files, desc="infer")):
        rows = predict_file(f, model, mel_extractor)
        all_rows.extend(rows)
        if (i + 1) % 50 == 0:
            elapsed = time.time() - start_t
            rate = (i + 1) / elapsed * 60
            eta = (len(test_files) - i - 1) / rate * 60 if rate > 0 else 0
            print(f"  [{i+1}/{len(test_files)}] {len(all_rows)} rows, {rate:.1f}/min, ETA {eta/60:.1f}min")
    elapsed = time.time() - start_t
    print(f"\nInference DONE in {elapsed/60:.1f}min")

# Build sub_df with proper columns (even if empty)
cols = ["row_id"] + PRIMARY_LABELS
if all_rows:
    sub_df = pd.DataFrame(all_rows)
    # Ensure column order
    sub_df = sub_df[cols]
else:
    sub_df = pd.DataFrame(columns=cols)

print(f"\nsub_df shape: {sub_df.shape}")
print(f"  columns count: {len(sub_df.columns)} (expected: {len(cols)})")


In [ ]:
# ============================================================
# Cell 8: PP (TopN N=1, paper 256 spec) — skip if empty
# ============================================================
if len(sub_df) == 0:
    print("⚠️ sub_df is empty, skipping PP (will use sample_submission as template)")
else:
    print(f"Applying TopN N=1 PP (FCS_TOP_K={CFG.FCS_TOP_K}, FCS_POWER={CFG.FCS_POWER})...")
    # Extract file_id from row_id
    sub_df["_file_id"] = sub_df["row_id"].apply(lambda x: x.rsplit("_", 1)[0])

    prob_cols = [c for c in sub_df.columns if c in PRIMARY_LABELS]
    preds_array = sub_df[prob_cols].values.astype(np.float32)
    file_ids = sub_df["_file_id"].values

    unique_files = pd.unique(file_ids)
    file_to_idx = {f: i for i, f in enumerate(unique_files)}
    file_idx_arr = np.array([file_to_idx[f] for f in file_ids])

    for f_idx in range(len(unique_files)):
        mask = (file_idx_arr == f_idx)
        file_preds = preds_array[mask]
        if CFG.FCS_TOP_K == 1:
            file_max = file_preds.max(axis=0)
        else:
            sorted_preds = np.sort(file_preds, axis=0)
            file_max = sorted_preds[-CFG.FCS_TOP_K:].mean(axis=0)
        scale = np.power(file_max, CFG.FCS_POWER)
        preds_array[mask] = file_preds * scale[np.newaxis, :]

    for j, col in enumerate(prob_cols):
        sub_df[col] = preds_array[:, j]

    sub_df = sub_df.drop(columns=["_file_id"])
    print(f"PP done. preds range: [{preds_array.min():.4f}, {preds_array.max():.4f}]")


In [ ]:
# ============================================================
# Cell 9: Save submission.csv (handle empty case)
# ============================================================
sample_sub_path = COMP_ROOT / "sample_submission.csv"
sample = pd.read_csv(sample_sub_path)
expected_cols = sample.columns.tolist()

if len(sub_df) == 0:
    print("⚠️ Empty sub_df, using sample_submission.csv as template (all zeros)")
    # Use sample_submission.csv structure (it has row_ids for test set)
    sub_df = sample.copy()
    print(f"  sample rows: {len(sub_df)}")
else:
    # Reorder columns to match sample
    sub_df = sub_df[expected_cols]
    print(f"Column order matched to sample_submission.csv ({len(expected_cols)} cols)")

OUT_PATH = Path("/kaggle/working/submission.csv")
sub_df.to_csv(OUT_PATH, index=False)
print(f"\nSaved: {OUT_PATH}")
print(f"  shape: {sub_df.shape}")
print(f"  size: {OUT_PATH.stat().st_size/1e6:.1f} MB")
print(f"\nFirst 3 rows:")
print(sub_df.head(3).iloc[:, :6])
